# Pipeline — Árbol de Decisión
### Módulo 3 · Tema 4.1

---
## ¿Qué hace un Árbol de Decisión?
Hace preguntas sucesivas del tipo `¿radio > 15?` hasta llegar a una decisión.
Construye un árbol de preguntas que divide los datos en grupos cada vez más puros.

```
          ¿radio > 15?
         /            \
       SÍ              NO
    ¿área > 700?    → Benigno
     /        \
  Maligno   Benigno
```

## Ventajas vs desventajas
```
✅ Muy interpretable — puedes visualizar las decisiones
✅ No necesita escalar los datos
✅ Maneja datos mixtos (números y categorías)
❌ Tiende a overfitting si crece demasiado
   → Control: max_depth (limita la profundidad)
```

## Parámetros clave
```
max_depth  → profundidad máxima. Más bajo = más simple = menos overfitting
min_samples_split → mínimo de muestras para dividir un nodo
```

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports
# ═══════════════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree            import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import accuracy_score, confusion_matrix, classification_report

RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

print("Imports listos ✅")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Cargar datos
# ← CAMBIA ESTE BLOQUE por tu dataset
# ═══════════════════════════════════════════════════════════════
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y = dataset.target
CLASES = list(dataset.target_names)

# Dividir — el árbol NO necesita escalar, pero lo incluimos por consistencia
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.25, stratify=Y, random_state=42
)

# El árbol puede trabajar con datos sin escalar, pero escalamos por consistencia
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3 — Probar diferentes profundidades
# Para encontrar el max_depth óptimo antes de elegir uno
# ═══════════════════════════════════════════════════════════════
profundidades = range(1, 10)
acc_train_vals = []
acc_test_vals  = []

for depth in profundidades:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train_sc, y_train)
    acc_train_vals.append(accuracy_score(y_train, m.predict(X_train_sc)))
    acc_test_vals.append(accuracy_score(y_test,  m.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(profundidades, acc_train_vals, 'b-o', label='Train', linewidth=2)
ax.plot(profundidades, acc_test_vals,  'r-o', label='Test',  linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Profundidad del Árbol')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mejor_depth = profundidades[acc_test_vals.index(max(acc_test_vals))]
print(f"Mejor max_depth para test: {mejor_depth}")
print()
print("Si train >> test → OVERFITTING (árbol demasiado profundo)")
print("Si ambas son bajas → UNDERFITTING (árbol demasiado simple)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Entrenar con el mejor max_depth
# ═══════════════════════════════════════════════════════════════
MAX_DEPTH = mejor_depth  # ← o cambia manualmente si lo prefieres

modelo = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=42)
modelo.fit(X_train_sc, y_train)

Y_pred = modelo.predict(X_test_sc)
acc    = accuracy_score(y_test, Y_pred)

print(f"Árbol (max_depth={MAX_DEPTH}) — Accuracy: {acc:.4f} ({acc*100:.1f}%)\n")
print(classification_report(y_test, Y_pred, target_names=CLASES))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5 — Visualizar el árbol
# La gran ventaja del árbol: puedes ver exactamente qué decisiones toma
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    modelo,
    feature_names=X.columns.tolist(),
    class_names=CLASES,
    filled=True,      # colorea según clase dominante en cada nodo
    rounded=True,
    ax=ax
)
ax.set_title(f'Árbol de Decisión (max_depth={MAX_DEPTH})')
plt.tight_layout()
plt.show()

# Importancia de características
# Cuánto usó el árbol cada columna para tomar decisiones
importancias = pd.Series(
    modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.head(10).plot(kind='barh', ax=ax, color='#3B8BD4')
ax.set_title('Top 10 características más usadas por el árbol')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 — Matriz de confusión
# ═══════════════════════════════════════════════════════════════
cm = confusion_matrix(y_test, Y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[f'Pred {c}' for c in CLASES],
    yticklabels=[f'Real {c}' for c in CLASES],
    ax=ax
)
ax.set_title(f'Matriz de Confusión — {acc*100:.1f}%')
plt.tight_layout()
plt.show()